In [5]:
pip install git+https://github.com/foundation-model-stack/foundation-model-stack.git

  Cloning https://github.com/foundation-model-stack/foundation-model-stack.git to /tmp/pip-req-build-oye1jhqx
  Running command git clone --filter=blob:none --quiet https://github.com/foundation-model-stack/foundation-model-stack.git /tmp/pip-req-build-oye1jhqx
  Resolved https://github.com/foundation-model-stack/foundation-model-stack.git to commit dc58ad573e4394220aeb7eef56df95a246b74334
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.1 MB/s eta 0:00:00
   ━━━━━

In [8]:
import torch
from fms.models.bamba import Bamba, BambaConfig

In [13]:
# TESTING HOW TO INITIALIZE BAMBA

# - mamba_expand = 1.0 ensures that 1.0 * emb_dim == nheads * head_dim (64 == 4 * 16)
# - n_groups is set to 1 so that nheads // n_groups = 4
# - chunk_size is set equal to the sequence length (10) to avoid any padding issues
test_config = BambaConfig(
    src_vocab_size=100,
    emb_dim=64,
    nheads=4,              # number of attention heads
    kvheads=4,             # for simplicity, set equal to nheads
    head_dim=16,           # head dimension such that 4 * 16 = 64
    nlayers=2,
    mamba_n_heads=4,       # SSM heads matching nheads
    state_size=32,         # smaller state size for SSM
    conv_kernel=3,
    mamba_expand=1.0,      # critical: 1.0 * 64 == 4 * 16
    p_dropout=0.0,         # disable dropout for testing
    attn_layer_indices=[1],# use an attention layer in one of the blocks (others use SSM)
    max_expected_seq_len=512,
    chunk_size=10,         # set chunk_size equal to the sequence length to avoid padding issues
    n_groups=1             # set n_groups so that nheads//n_groups is not zero
)

# Instantiate the Bamba model using the test config
model = Bamba(config=test_config)

# Initialize parameters and do post-init steps (e.g. setting up RoPE caches)
model.reset_parameters()
model.post_init()

# Create a dummy input tensor with shape [batch_size, seq_len]
dummy_input = torch.randint(0, test_config.src_vocab_size, (1, 10))

# Run a forward pass through the model
# Depending on the caching flag, forward() might return a tuple (predictions, cache)
output = model(dummy_input)
if isinstance(output, tuple):
    preds, cache = output
else:
    preds = output

print("Output shape:", preds.shape)

Output shape: torch.Size([1, 10, 100])


In [ ]:
# If we want to overwrite the SSM block

#from fms.modules import ssm

# Define the new SSM
#class MyNewSSM(ssm.SSM):
    #def forward(self, input_states, mask, past_key_value_state=None, cache_position=None, **kwargs):
        # the modified implementation
        #print("Using MyNewSSM")
        #return super().forward(input_states, mask, past_key_value_state, cache_position, **kwargs)

# Overwrite the original SSM with the new version
#ssm.SSM = MyNewSSM

# Alternatively, we can create a new module ssm_chunked and rewrite Bamba with it

In [ ]:
# FULL ssm.py CODE

"""
from typing import Optional

import torch
import torch.nn as nn

from fms.utils.activation import str_to_activation


def pad_tensor_by_size(input_tensor: torch.Tensor, pad_size: int):
    """
    #Padding x tensor with `pad_size` on the seq_len dim (dim=1)

    #Assumes that we only have tensors of either size 4 or 3
    """
    pad_shape = (
        (0, 0, 0, 0, 0, pad_size, 0, 0)
        if len(input_tensor.shape) == 4
        else (0, 0, 0, pad_size, 0, 0)
    )

    return torch.nn.functional.pad(input_tensor, pad_shape, mode="constant", value=0)


def reshape_into_chunks(input_tensor, pad_size, chunk_size):
    """
    #Padding input_tensor with `pad_size` on the seq_len dim (dim=1) and
    #simultaneously splitting it into chunk sequences.

    #Assumes that we only have tensors of either size 4 or 3
    """
    # [bsz, seq_len, ...] -> [bsz, seq_len multiple of chunk_size, ...]
    input_tensor = pad_tensor_by_size(input_tensor, pad_size)

    if len(input_tensor.shape) == 3:
        # [bsz, seq_len multiple of chunk_size, nheads] -> [bsz, -1, chunk_size, nheads]
        return input_tensor.reshape(
            input_tensor.shape[0], -1, chunk_size, input_tensor.shape[2]
        )
    else:
        # [bsz, seq_len multiple of chunk_size, nheads, head_dim or state_size] -> [bsz, -1, chunk_size, nheads, head_dim or state_size]
        return input_tensor.reshape(
            input_tensor.shape[0],
            -1,
            chunk_size,
            input_tensor.shape[2],
            input_tensor.shape[3],
        )


def segment_sum(input_tensor):
    """
    #More stable segment sum calculation. Uses cumulative sums and masking instead of direct subtractions.
    """
    chunk_size = input_tensor.size(-1)
    # 1. expand input tensor to have an additional dimension and repeat along that dimension
    # [..., chunk_size] -> [..., chunk_size, chunk_size]
    input_tensor = input_tensor[..., None].expand(*input_tensor.size(), chunk_size)
    # 2. create a lower triangular mask with the diagonal set to 0 to 0 out elements above diag
    mask = torch.tril(
        torch.ones(
            chunk_size, chunk_size, device=input_tensor.device, dtype=torch.bool
        ),
        diagonal=-1,
    )
    input_tensor = input_tensor.masked_fill(~mask, 0)
    # 3. compute actual cumsum
    tensor_segsum = torch.cumsum(input_tensor, dim=-2)

    # 4. apply mask to keep only the lower triangular part of the cumulative sum result (incl diagonal this time)
    mask = torch.tril(
        torch.ones(
            chunk_size, chunk_size, device=input_tensor.device, dtype=torch.bool
        ),
        diagonal=0,
    )
    tensor_segsum = tensor_segsum.masked_fill(~mask, -torch.inf)
    return tensor_segsum


class RMSNormGated(nn.Module):
    def __init__(self, emb_dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(emb_dim))
        self.variance_epsilon = eps

    def forward(self, hidden_states, gate=None):
        input_dtype = hidden_states.dtype
        hidden_states = hidden_states.to(torch.float32)

        if gate is not None:
            hidden_states = hidden_states * nn.functional.silu(gate.to(torch.float32))
        variance = hidden_states.pow(2).mean(-1, keepdim=True)
        hidden_states = hidden_states * torch.rsqrt(variance + self.variance_epsilon)

        return self.weight * hidden_states.to(input_dtype)


class SSMCacheUnit:
    def __init__(
        self,
        emb_dim: int,
        nheads: int,
        head_dim: int,
        conv_kernel,
        expand: float,
        n_groups: int,
        state_size: int,
        batch_size: int,
        dtype: torch.dtype,
        device: Optional[str] = None,
    ):
        self.seqlen_offset = 0
        self.dtype = dtype
        self.conv_kernel_size = conv_kernel
        self.intermediate_size = int(expand * emb_dim)
        self.has_previous_state = False

        self.conv_state = torch.zeros(
            batch_size,
            self.intermediate_size + 2 * n_groups * state_size,
            self.conv_kernel_size,
            device=device,
            dtype=dtype,
        )
        self.ssm_state = torch.zeros(
            batch_size, nheads, head_dim, state_size, device=device, dtype=dtype
        )

    def update_conv_state(
        self, new_conv_state: torch.Tensor, cache_position: torch.Tensor
    ):
        conv_state = self.conv_state
        cache_position = cache_position.clamp(0, self.conv_kernel_size - 1)

        conv_state = conv_state.roll(shifts=-1, dims=-1)
        conv_state[:, :, cache_position] = new_conv_state.to(conv_state.device)
        self.conv_state.zero_()
        self.conv_state += conv_state
        return self.conv_state


def apply_mask_to_padding_states(hidden_states, attention_mask):
    """
    #Tunes out the hidden states for padding tokens, see https://github.com/state-spaces/mamba/issues/66
    """
    if (
        attention_mask is not None
        and attention_mask.shape[1] > 1
        and attention_mask.shape[0] > 1
    ):
        dtype = hidden_states.dtype
        hidden_states = (hidden_states * (attention_mask[:, -1, :, None] == 0)).to(
            dtype
        )

    return hidden_states


class SSM(nn.Module):
    def __init__(
        self,
        nheads: int,
        emb_dim: int,
        state_size: int,
        conv_kernel: int,
        expand: float,
        use_bias: bool,
        use_conv_bias: bool,
        activation_fn: str,
        norm_eps: float,
        n_groups: int,
        head_dim: int,
        chunk_size: int,
    ):
        super(SSM, self).__init__()
        self.nheads = nheads
        self.emb_dim = emb_dim
        self.ssm_state_size = state_size
        self.conv_kernel_size = conv_kernel
        self.intermediate_size = int(expand * self.emb_dim)
        self.use_conv_bias = use_conv_bias
        self.activation = activation_fn
        self.act = str_to_activation(activation_fn)

        self.layer_norm_epsilon = norm_eps

        self.n_groups = n_groups
        self.head_dim = head_dim
        self.chunk_size = chunk_size

        self.conv_dim = self.intermediate_size + 2 * self.n_groups * self.ssm_state_size
        self.conv1d = nn.Conv1d(
            in_channels=self.conv_dim,
            out_channels=self.conv_dim,
            bias=use_conv_bias,
            kernel_size=conv_kernel,
            groups=self.conv_dim,
            padding=conv_kernel - 1,
        )

        # projection of the input hidden states
        projection_size = self.intermediate_size + self.conv_dim + self.nheads
        self.in_proj = nn.Linear(
            self.emb_dim,
            projection_size,
            bias=use_bias,
        )
        # selective projection used to make dt, B and C input dependant

        # time step projection (discretization)
        # instantiate once and copy inv_dt in init_weights of PretrainedModel
        self.dt_bias = nn.Parameter(torch.ones(self.nheads))

        # S4D real initialization. These are not discretized!
        # The core is to load them, compute the discrete states, then write the updated state. Keeps the memory bounded
        A = torch.arange(1, self.nheads + 1)
        self.A_log = nn.Parameter(torch.log(A))
        self.norm = RMSNormGated(self.intermediate_size, eps=self.layer_norm_epsilon)
        self.D = nn.Parameter(torch.ones(self.nheads))

        self.time_step_limit = (0.0, float("inf"))
        self.time_step_min = 0.001
        self.time_step_max = 0.1
        self.out_proj = nn.Linear(self.intermediate_size, self.emb_dim, bias=use_bias)

    def forward(
        self,
        input_states,
        mask,
        past_key_value_state: Optional[SSMCacheUnit] = None,
        cache_position: Optional[torch.Tensor] = None,
        **kwargs,
    ):
        batch_size, seq_len, _ = input_states.shape
        dtype = input_states.dtype

        # 1. Gated MLP's linear projection
        input_states = apply_mask_to_padding_states(input_states, mask)
        projected_states = self.in_proj(input_states)
        gate, hidden_states_B_C, dt = projected_states.split(
            [self.intermediate_size, self.conv_dim, self.nheads], dim=-1
        )

        use_precomputed_states = (
            past_key_value_state is not None
            and past_key_value_state.has_previous_state
            and seq_len == 1
            and past_key_value_state.conv_state.shape[0]
            == past_key_value_state.ssm_state.shape[0]
            == batch_size
            and cache_position is not None
            # removed cache_position 0 check in favor of has_previous_state to fix torch compile
        )

        # 2. Convolution sequence transformation
        if use_precomputed_states:
            past_key_value_state.conv_state = past_key_value_state.conv_state.roll(  # type: ignore[union-attr]
                shifts=-1, dims=-1
            )
            past_key_value_state.conv_state[:, :, -1] = hidden_states_B_C[:, 0, :].to(  # type: ignore[union-attr]
                past_key_value_state.conv_state.device  # type: ignore[union-attr]
            )

            # We need to guarantee that anything regarding the cache is on the same device
            conv_states = past_key_value_state.conv_state.to(  # type: ignore[union-attr]
                device=self.conv1d.weight.device
            )

            hidden_states_B_C = torch.sum(
                conv_states * self.conv1d.weight.squeeze(1), dim=-1
            )
            if self.use_conv_bias:
                hidden_states_B_C = hidden_states_B_C + self.conv1d.bias
            hidden_states_B_C = self.act(hidden_states_B_C)
        else:
            hidden_states_B_C_transposed = hidden_states_B_C.transpose(1, 2)

            # Init cache
            if past_key_value_state is not None:
                conv_states = nn.functional.pad(
                    hidden_states_B_C_transposed,
                    (self.conv_kernel_size - hidden_states_B_C_transposed.shape[-1], 0),
                )
                past_key_value_state.conv_state.copy_(conv_states)

            hidden_states_B_C = self.act(
                self.conv1d(hidden_states_B_C_transposed)[..., :seq_len].transpose(1, 2)
            )

        hidden_states_B_C = apply_mask_to_padding_states(hidden_states_B_C, mask)
        hidden_states, B, C = torch.split(
            hidden_states_B_C,
            [
                self.intermediate_size,
                self.n_groups * self.ssm_state_size,
                self.n_groups * self.ssm_state_size,
            ],
            dim=-1,
        )

        # 3. SSM transformation
        A = -torch.exp(self.A_log.float())  # [num_heads]
        if use_precomputed_states:
            # We need to guarantee that anything regarding the cache is on the same device
            cache_device = past_key_value_state.ssm_state.device  # type: ignore[union-attr]

            # Note: there is no need to pad parameter matrices here, as there is just one new token
            # for batched generation
            dt = dt[:, 0, :][:, None, ...]
            dt = dt.transpose(1, 2).expand(batch_size, dt.shape[-1], self.head_dim)
            # [num_heads] -> [num_heads, head_dim]
            dt_bias = self.dt_bias[..., None].expand(
                self.dt_bias.shape[0], self.head_dim
            )

            dt = torch.nn.functional.softplus(dt + dt_bias.to(dt.dtype))
            dt = torch.clamp(dt, self.time_step_limit[0], self.time_step_limit[1])
            A = (
                A[..., None, None]
                .expand(self.nheads, self.head_dim, self.ssm_state_size)
                .to(dtype=torch.float32)
            )
            # [bsz, num_heads, head_dim, state_size]
            dA = (torch.exp(dt[..., None] * A)).to(device=cache_device)

            # Discretize B
            # [bsz, n_groups * state_size] -> [bsz, n_groups, 1, state_size] ->
            # -> [bsz, n_groups, group to head repetition factor, state_size] -> [bsz, num_heads, state_size]
            B = B.reshape(batch_size, self.n_groups, -1)[..., None, :]
            B = B.expand(
                batch_size, self.n_groups, self.nheads // self.n_groups, B.shape[-1]
            ).contiguous()
            B = B.reshape(batch_size, -1, B.shape[-1])
            # [bsz, num_heads, head_dim, state_size]
            dB = dt[..., None] * B[..., None, :]

            # Discretize x into dB
            # [bsz, intermediate_size] -> [bsz, num_heads, head_dim]
            hidden_states = hidden_states.reshape(batch_size, -1, self.head_dim)
            dBx = (dB * hidden_states[..., None]).to(device=cache_device)

            # State calculation
            past_key_value_state.ssm_state.copy_(  # type: ignore[union-attr]
                past_key_value_state.ssm_state * dA + dBx  # type: ignore[union-attr]
            )

            # Subsequent output
            # [bsz, n_groups * state_size] -> [bsz, num_heads, state_size]
            C = C.reshape(batch_size, self.n_groups, -1)[..., None, :]
            C = C.expand(
                batch_size, self.n_groups, self.nheads // self.n_groups, C.shape[-1]
            ).contiguous()
            C = C.reshape(batch_size, -1, C.shape[-1])
            # [bsz, num_heads, head_dim]

            ssm_states = past_key_value_state.ssm_state.to(  # type: ignore[union-attr]
                device=C.device, dtype=C.dtype
            )  # Shape: [b, h, d, n]
            # Reshape ssm_states to merge the first two dimensions
            ssm_states_reshaped = ssm_states.view(
                batch_size * self.nheads, self.head_dim, self.ssm_state_size
            )  # Shape: [b*h, d, n]
            C_reshaped = C.view(
                batch_size * self.nheads, self.ssm_state_size, 1
            )  # Shape: [b*h, n, 1]
            y = torch.bmm(ssm_states_reshaped, C_reshaped)
            y = y.view(batch_size, self.nheads, self.head_dim)

            # D skip connection
            # [num_heads] -> [num_heads, head_dim]
            D = self.D[..., None].expand(self.D.shape[0], self.head_dim)
            y = (y + hidden_states * D).to(y.dtype)

            # [bsz, num_heads, head_dim] -> [bsz, 1, intermediate_size]
            y = y.reshape(batch_size, -1)[:, None, ...]
        else:
            # begin ssd naive implementation without einsums
            dt = nn.functional.softplus(dt + self.dt_bias)
            dt = torch.clamp(dt, self.time_step_limit[0], self.time_step_limit[1])
            hidden_states = hidden_states.reshape(
                batch_size, seq_len, -1, self.head_dim
            ).float()
            B = B.reshape(batch_size, seq_len, -1, self.ssm_state_size).float()
            C = C.reshape(batch_size, seq_len, -1, self.ssm_state_size).float()
            B = B.repeat(1, 1, self.nheads // self.n_groups, 1)
            C = C.repeat(1, 1, self.nheads // self.n_groups, 1)
            pad_size = (self.chunk_size - seq_len % self.chunk_size) % self.chunk_size

            D_residual = self.D[..., None] * pad_tensor_by_size(hidden_states, pad_size)

            # Discretize x and A
            hidden_states = hidden_states * dt[..., None]
            A = A.to(hidden_states.dtype) * dt

            # Rearrange into blocks/chunks
            hidden_states, A, B, C = [
                reshape_into_chunks(t, pad_size, self.chunk_size)
                for t in (hidden_states, A, B, C)
            ]

            # [bsz, -1, chunk_size, num_heads] -> [bsz, num_heads, -1, chunk_size]
            A = A.permute(0, 3, 1, 2)
            A_cumsum = torch.cumsum(A, dim=-1)

            # 1. Compute the output for each intra-chunk (diagonal blocks)
            # This is the analog of a causal mask
            L = torch.exp(segment_sum(A))

            # Contraction of C and B to get G (attention-weights like)
            G_intermediate = (
                C[:, :, :, None, :, :] * B[:, :, None, :, :, :]
            )  # shape: (b, c, l, s, h, n)
            G = G_intermediate.sum(dim=-1)  # shape: (b, c, l, s, h)

            # Compute M, equivalent to applying attention mask to weights
            M_intermediate = G[..., None] * L.permute(0, 2, 3, 4, 1)[..., None]
            M = M_intermediate.sum(dim=-1)

            # Compute Y_diag (apply to values)
            Y_diag = (M[..., None] * hidden_states[:, :, None]).sum(dim=3)

            # 2. Compute the state for each intra-chunk
            # (right term of low-rank factorization of off-diagonal blocks; B terms)
            decay_states = torch.exp((A_cumsum[:, :, :, -1:] - A_cumsum))
            B_decay = B * decay_states.permute(0, -2, -1, 1)[..., None]
            states = (B_decay[..., None, :] * hidden_states[..., None]).sum(dim=2)

            # 3. Compute the inter-chunk SSM recurrence; produces correct SSM states at chunk boundaries
            # (middle term of factorization of off-diag blocks; A terms)
            if use_precomputed_states:
                previous_states = past_key_value_state.ssm_state[:, None, ...].to(
                    device=states.device
                )
            else:
                previous_states = torch.zeros_like(states[:, :1])
            states = torch.cat([previous_states, states], dim=1)
            decay_chunk = torch.exp(
                segment_sum(nn.functional.pad(A_cumsum[:, :, :, -1], (1, 0)))
            )
            decay_chunk = decay_chunk.transpose(1, 3)
            new_states = (decay_chunk[..., None, None] * states[:, :, None, ...]).sum(
                dim=1
            )
            states, ssm_state = new_states[:, :-1], new_states[:, -1]

            # 4. Compute state -> output conversion per chunk
            # (left term of low-rank factorization of off-diagonal blocks; C terms)
            state_decay_out = torch.exp(A_cumsum)
            C_times_states = C[..., None, :] * states[:, :, None, ...]
            state_decay_out_permuted = state_decay_out.permute(0, 2, 3, 1)
            Y_off = C_times_states.sum(-1) * state_decay_out_permuted[..., None]

            # Add output of intra-chunk and inter-chunk terms (diagonal and off-diagonal blocks)
            y = Y_diag + Y_off
            # [bsz, -1, self.chunk_size, num_heads, head_dim] -> [bsz, (padded) seq_len, num_heads, head_dim]
            y = y.reshape(batch_size, -1, self.nheads, self.head_dim)

            y = y + D_residual
            # Cutting off padded chunks
            if pad_size > 0:
                y = y[:, :seq_len, :, :]
            y = y.reshape(batch_size, seq_len, -1)

            # Init cache
            if ssm_state is not None and past_key_value_state is not None:
                past_key_value_state.ssm_state.copy_(ssm_state)

        scan_output = self.norm(y, gate)

        # end ssd naive

        # 4. Final linear projection
        contextualized_states = self.out_proj(
            scan_output.to(dtype)
        )  # [batch, seq_len, hidden_size]
        return contextualized_states, past_key_value_state
"""